# Matter vs antimatter in 1 GB of LHCb open data

*OpSciHack 2026 · "Open Datasets from CERN" · F. L. Redi*

In 2013 LHCb published the first evidence that **B⁺ and B⁻ mesons decay differently** into three kaons
([PRL 111 (2013) 101801](https://doi.org/10.1103/PhysRevLett.111.101801)). The 2011 data behind that measurement are public,
so you can recompute the headline number yourself in a few minutes.

* **Data:** LHCb, 2011, 7 TeV proton–proton collisions, B± → h± h⁺ h⁻ candidates, about 8.6 million events in 2 ROOT files (1.1 GB)
* **Record:** <https://opendata.cern.ch/record/4900>, DOI [10.7483/OPENDATA.LHCB.AOF7.JH09](https://doi.org/10.7483/OPENDATA.LHCB.AOF7.JH09), CC0 licence
* **Runs on:** Google Colab, Binder, SWAN or a laptop with Python 3

**What you will do:** pick events where all three particles look like kaons, compute the mass of their parent,
then count how many come from B⁺ and how many from B⁻.

In [ ]:
# Setup: uproot reads ROOT files without installing ROOT
%pip install -q uproot awkward

In [ ]:
# Download the two data files (about 1.1 GB in total; one file is enough for a quick look)
!curl -sSLO https://opendata.cern.ch/eos/opendata/lhcb/AntimatterMatters2017/data/B2HHH_MagnetUp.root
!curl -sSLO https://opendata.cern.ch/eos/opendata/lhcb/AntimatterMatters2017/data/B2HHH_MagnetDown.root

## 1. Select three-kaon candidates and compute their mass
Each event has three tracks `H1`, `H2` and `H3`, with momenta (`PX`, `PY`, `PZ`), charge, and the detector's probability that each track is a kaon (`ProbK`) or a pion (`ProbPi`).

In [ ]:
import uproot, numpy as np, matplotlib.pyplot as plt

M_K = 493.677  # charged-kaon mass in MeV/c^2
files = ["B2HHH_MagnetUp.root", "B2HHH_MagnetDown.root"]
branches = [f"H{i}_{v}" for i in (1, 2, 3) for v in ("PX", "PY", "PZ", "ProbK", "ProbPi", "Charge", "isMuon")]

masses, charges, m2low = [], [], []
for chunk in uproot.iterate([f + ":DecayTree" for f in files], branches, step_size="200 MB", library="np"):
    # 1. keep events where all three tracks look like kaons (and not muons)
    ok = np.ones(len(chunk["H1_PX"]), bool)
    for i in (1, 2, 3):
        ok &= (chunk[f"H{i}_ProbK"] > 0.6) & (chunk[f"H{i}_ProbPi"] < 0.4) & (chunk[f"H{i}_isMuon"] == 0)
    p = [np.stack([chunk[f"H{i}_P{c}"][ok] for c in "XYZ"], axis=1) for i in (1, 2, 3)]
    q = [chunk[f"H{i}_Charge"][ok] for i in (1, 2, 3)]
    E = [np.sqrt((pi ** 2).sum(1) + M_K ** 2) for pi in p]
    # 2. invariant mass of the three kaons = mass of the parent B meson
    Etot, ptot = sum(E), sum(p)
    masses.append(np.sqrt(Etot ** 2 - (ptot ** 2).sum(1)))
    charges.append(q[0] + q[1] + q[2])
    # 3. the two opposite-charge K+K- pairs; keep the lighter one (for step 5)
    def m2(a, b):
        return ((E[a] + E[b]) ** 2 - ((p[a] + p[b]) ** 2).sum(1)) / 1e6  # GeV^2/c^4
    pairs = np.stack([np.where(q[a] != q[b], m2(a, b), np.inf) for a, b in ((0, 1), (0, 2), (1, 2))], axis=1)
    m2low.append(pairs.min(axis=1))

mass, charge, m2low = np.concatenate(masses), np.concatenate(charges), np.concatenate(m2low)
print(f"{len(mass):,} three-kaon candidates")

## 2. Look at the mass peak

In [ ]:
bins = np.arange(5150, 5452, 6)
plt.figure(figsize=(8, 5))
plt.hist(mass[charge == +1], bins, histtype="step", lw=2, label="B+")
plt.hist(mass[charge == -1], bins, histtype="step", lw=2, label="B-")
plt.xlabel("Mass of the three kaons [MeV/c$^2$]"); plt.ylabel("Candidates per 6 MeV/c$^2$")
plt.legend(); plt.show()

## 3. Count B⁺ and B⁻ and compute the asymmetry
This is a *raw* asymmetry: unlike the paper, we do not correct for the small differences in how B⁺ and B⁻ are produced and detected (the paper does, using B± → J/ψ K±).

In [ ]:
# 4. count B+ and B- in the signal window, subtracting the flat background seen in a sideband
signal = (mass > 5250) & (mass < 5310)          # 60 MeV wide
sideband = (mass > 5400) & (mass < 5700)        # 300 MeV wide, no signal here
N = {}
for name, q in (("B+", +1), ("B-", -1)):
    n_win = np.sum(signal & (charge == q))
    n_bkg = np.sum(sideband & (charge == q)) * 60 / 300
    N[name] = n_win - n_bkg
    print(f"{name}: {n_win} in window, ~{n_bkg:.0f} background -> {N[name]:.0f} signal")
A = (N["B-"] - N["B+"]) / (N["B-"] + N["B+"])
dA = np.sqrt((1 - A ** 2) / (N["B-"] + N["B+"]))
print(f"raw asymmetry = {A:+.3f} +- {dA:.3f}  (LHCb 2013, PRL 111, 101801: -0.043 +- 0.009 +- 0.003 +- 0.007)")

## 4. Where in the decay does the difference sit?
LHCb found that the asymmetry is concentrated in particular regions of the decay. Here we split the signal by the mass of the lighter K⁺K⁻ pair.

In [ ]:
edges = [1.0, 1.5, 2, 3, 5, 8, 12]   # GeV^2/c^4
A_loc, err = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    s = signal & (m2low >= lo) & (m2low < hi)
    npl, nmi = np.sum(s & (charge == 1)), np.sum(s & (charge == -1))
    a = (nmi - npl) / (nmi + npl); A_loc.append(100 * a); err.append(100 * np.sqrt((1 - a * a) / (nmi + npl)))
labels = [f"{lo:g}-{hi:g}" for lo, hi in zip(edges[:-1], edges[1:])]
plt.figure(figsize=(8, 5))
plt.bar(labels, A_loc, yerr=err, capsize=5); plt.axhline(0, color="k", lw=1)
plt.xlabel("Lower K$^+$K$^-$ mass$^2$ [GeV$^2$/c$^4$]"); plt.ylabel("B$^-$ vs B$^+$ asymmetry [%]"); plt.show()

## Things to try
1. Tighten or loosen the kaon cuts (`ProbK > 0.6`, `ProbPi < 0.4`). How do the yields and the asymmetry change?
2. Swap the cuts so that all three tracks are **pions**, and use the pion mass (139.570 MeV/c²). Do you see a peak? (This is B± → π±π⁺π⁻.)
3. Make a 2D "Dalitz plot" of the two K⁺K⁻ masses squared, separately for B⁺ and B⁻.
4. Compare the two magnet polarities (`MagnetUp` vs `MagnetDown`). Detector effects should flip, but physics should not.

**Want other decays or variables?** The [LHCb Ntupling Service](https://opendata.cern.ch/) (Feb 2026) lets you request custom ntuples from Run 1 and Run 2 data without installing any LHCb software. Questions go to the [CERN Open Data Forum](https://opendata-forum.cern.ch/c/lhcb/13).